# 02g — Precision-first pipeline (alternative to 02b → 02e → 02f)

End-to-end re-run of feature selection, hyper-parameter tuning, and
post-processing optimised for **average precision** (AP) instead of
balanced accuracy / AUC. The intent is to verify whether a coherent
precision-first optimisation beats the AUC-trained model with a
precision-optimised post-processing layer (02e + 02f).

| Section | What | Optimised for |
|---------|------|---------------|
| A | Load kitchen-sink data | – |
| B | Variance + correlation pre-filter (ρ=0.95) | – |
| C | 4-method selection with AP scoring | AP |
| D | Winner pick from candidate subsets | AP |
| E | SMOTE strategy comparison | AP |
| F | Optuna hyperparameter tuning | AP |
| G | Threshold tuning | F1 |
| H | Top-K per match + position filter | F1 + filter |
| I | Compare 02g vs 02e+02f | both metrics |

**The 02b / 02e / 02f notebooks remain untouched.** 02g is an
alternative configuration tracked separately in `models/precision_first/`.


## Setup

In [1]:
"""Imports."""
from __future__ import annotations
import sys, warnings, pickle, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT: Path = Path.cwd().parent if Path.cwd().name == "models" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [2]:
"""Modelling stack."""
import xgboost as xgb
import optuna
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.combine import SMOTETomek
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.feature_selection import RFECV
from sklearn.metrics import (
    roc_auc_score, balanced_accuracy_score, average_precision_score,
    brier_score_loss, log_loss, precision_score, recall_score, f1_score,
    fbeta_score, confusion_matrix, classification_report,
)
from sklearn.isotonic import IsotonicRegression
from boruta import BorutaPy
optuna.logging.set_verbosity(optuna.logging.WARNING)


## Section A — Load same kitchen-sink data as 02b

We load the merged feature panel (~117 candidates) from the same three
sources as 02b, apply the same leakage-removal preprocessing.


In [4]:
"""Load all three sources + merge."""
DATA_DIR = PROJECT_ROOT / "data"
FEATURES_DIR = PROJECT_ROOT / "features"

all_eng = pd.read_csv(FEATURES_DIR / "all_engineered_features.csv")
fe_teammate = pd.read_csv(FEATURES_DIR / "players_quarters_final_feature_engineered.csv")
main = pd.read_csv(DATA_DIR / "players_quarters_final.csv")

CHECKPOINT_REMAP = {
    "H1_15": "H1_15", "H1_30": "H1_30", "H1_45": "H1_45",
    "H2_60": "H2_15", "H2_75": "H2_30", "H2_90": "H2_45",
}
fe_teammate = fe_teammate.copy()
fe_teammate["checkpoint"] = fe_teammate["checkpoint"].map(CHECKPOINT_REMAP)

JOIN = ["player_appearance_id", "checkpoint"]
panel = main.merge(
    fe_teammate.drop(columns=[c for c in fe_teammate.columns if c in main.columns and c not in JOIN]),
    on=JOIN, how="inner",
).merge(
    all_eng.drop(columns=[c for c in all_eng.columns if c not in JOIN]),
    on=JOIN, how="left",
)

# Pre-process: drop minute_out (leakage), add mins_on_pitch_so_far.
panel["mins_on_pitch_so_far"] = (
    panel["checkpoint"].map({
        "H1_15": 15, "H1_30": 30, "H1_45": 45,
        "H2_15": 60, "H2_30": 75, "H2_45": 90, "ET1_15": 105,
    }) - panel["minute_in"]
).clip(lower=0)
panel["is_home_int"] = panel["is_home"].astype(bool).astype(int)
panel["subbed_int"] = panel["subbed"].astype(str).str.upper().eq("TRUE").astype(int)
position_dummies = pd.get_dummies(panel["position"], prefix="pos").astype(int)
panel = pd.concat([panel, position_dummies], axis=1)

TARGET = "scored_after"
GROUP = "fixture_id"
ID_COLS = ["player_appearance_id", "player_id", "fixture_id", "date",
           "checkpoint", "checkpoint_period", "checkpoint_min"]
LEAKAGE_COLS = ["minute_out"]
DROP_COLS = ID_COLS + LEAKAGE_COLS + ["position", "formation", "is_home", "subbed", TARGET]
candidate_features = [c for c in panel.columns
                       if c not in DROP_COLS and panel[c].dtype != "object"]
print(f"merged panel: {panel.shape}, target rate: {panel[TARGET].mean():.4f}")
print(f"candidate features: {len(candidate_features)}")


merged panel: (3114, 51), target rate: 0.0649
candidate features: 38


## Section B — Match-level split + pre-filter (same as 02b)

In [5]:
"""Match-level train/test split (identical to 02b)."""
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(panel, panel[TARGET], groups=panel[GROUP]))
train = panel.iloc[train_idx].reset_index(drop=True)
test = panel.iloc[test_idx].reset_index(drop=True)

X_train_full = train[candidate_features].fillna(0.0)
y_train = train[TARGET].astype(int)
g_train = train[GROUP].values
X_test_full = test[candidate_features].fillna(0.0)
y_test = test[TARGET].astype(int)
cp_train = train["checkpoint"].values
cp_test = test["checkpoint"].values
train_pos = train["position"].values
test_pos = test["position"].values

n_pos, n_neg = int(y_train.sum()), int((1 - y_train).sum())
spw = n_neg / max(n_pos, 1)
print(f"train: {X_train_full.shape}, fixtures={train[GROUP].nunique()}, "
      f"pos={n_pos}, neg={n_neg}, spw={spw:.2f}")
print(f"test:  {X_test_full.shape}, fixtures={test[GROUP].nunique()}, "
      f"pos={int(y_test.sum())}")


train: (2394, 38), fixtures=24, pos=173, neg=2221, spw=12.84
test:  (720, 38), fixtures=7, pos=29


In [6]:
"""Variance filter."""
def is_near_constant(s, threshold=0.99):
    if s.dropna().nunique() <= 1: return True
    return s.value_counts(normalize=True, dropna=False).iloc[0] >= threshold

near_const = [c for c in candidate_features if is_near_constant(X_train_full[c])]
features_var = [c for c in candidate_features if c not in near_const]
print(f"after variance filter: {len(features_var)}/{len(candidate_features)}")


after variance filter: 38/38


In [7]:
"""Correlation pre-prune at ρ=0.95 (identical to 02b)."""
y = y_train.values
r_with_target = pd.Series({
    c: abs(np.corrcoef(X_train_full[c], y)[0, 1]) if X_train_full[c].std() > 0 else 0.0
    for c in features_var
})
ordered = r_with_target.sort_values(ascending=False).index.tolist()

corr_abs = X_train_full[features_var].corr().abs()
kept = []
for f in ordered:
    bad = False
    for k in kept:
        if corr_abs.at[f, k] >= 0.95:
            bad = True; break
    if not bad: kept.append(f)
features_pre = kept
print(f"after rho=0.95 prune: {len(features_pre)}")

X_train_pre = X_train_full[features_pre]
X_test_pre = X_test_full[features_pre]


after rho=0.95 prune: 31


## Section C — Feature selection with AP scoring

The key change vs. 02b: we use **average precision** as the
cross-validation scoring metric for any selector that lets us choose
the metric. This nudges the procedure toward features that improve
precision-recall behaviour rather than overall ranking.

* **LASSO C-sweep** — `scoring='average_precision'`
* **Stability LASSO** — same C as the AP-best LASSO
* **BorutaPy** — uses tree importance natively (no scoring choice)
* **RFE-CV** — `scoring='average_precision'`


In [8]:
"""LASSO C-sweep with AP scoring."""
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train_pre)

C_values = [0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
sweep_rows = []
for C in C_values:
    aps = []
    for tr, va in GroupKFold(5).split(X_train_scaled, y_train, g_train):
        clf = LogisticRegression(
            penalty="l1", solver="liblinear", C=C,
            class_weight="balanced", max_iter=2000, random_state=RANDOM_SEED,
        )
        clf.fit(X_train_scaled[tr], y_train.iloc[tr])
        proba = clf.predict_proba(X_train_scaled[va])[:, 1]
        aps.append(average_precision_score(y_train.iloc[va], proba))
    nonzero = (np.abs(clf.coef_[0]) > 1e-8).sum()
    sweep_rows.append({"C": C, "cv_ap_mean": float(np.mean(aps)), "n_features": int(nonzero)})

sweep_df = pd.DataFrame(sweep_rows)
print(sweep_df.to_string(index=False))
best_C = float(sweep_df.loc[sweep_df["cv_ap_mean"].idxmax(), "C"])
print(f"best C (by AP): {best_C}")


   C  cv_ap_mean  n_features
0.05      0.1536          15
0.10      0.1496          19
0.20      0.1445          24
0.50      0.1411          28
1.00      0.1418          28
2.00      0.1401          29
best C (by AP): 0.05


In [9]:
"""LASSO selection at AP-best C."""
clf_lasso = LogisticRegression(
    penalty="l1", solver="liblinear", C=best_C,
    class_weight="balanced", max_iter=2000, random_state=RANDOM_SEED,
)
clf_lasso.fit(X_train_scaled, y_train)
lasso_features = pd.Series(np.abs(clf_lasso.coef_[0]), index=features_pre)
selection_lasso = lasso_features[lasso_features > 1e-8].sort_values(ascending=False).index.tolist()
print(f"LASSO@best-C selected {len(selection_lasso)} features")
print(lasso_features.sort_values(ascending=False).head(15).round(4).to_string())


LASSO@best-C selected 13 features
ratio_peak_speed            0.3502
formation_offensiveness     0.3142
pos_A                       0.2993
pos_D                       0.2670
subbed_int                  0.1257
is_home_int                 0.1248
jersey_number               0.1103
minute_in                   0.0927
ratio_distance              0.0823
last15_sprints              0.0729
cumul_hsr                   0.0381
last15_peak_speed           0.0315
ratio_shots_top_third       0.0282
last15_shots_under_press    0.0000
cumul_sprints               0.0000


In [10]:
"""Stability LASSO with bootstrap."""
N_BOOTSTRAPS = 50
bootstrap_picks = pd.DataFrame(0, index=features_pre, columns=range(N_BOOTSTRAPS))
rng = np.random.RandomState(RANDOM_SEED)

for b in range(N_BOOTSTRAPS):
    idx_boot = rng.choice(len(X_train_scaled), size=len(X_train_scaled), replace=True)
    Xb = X_train_scaled[idx_boot]
    yb = y_train.iloc[idx_boot].reset_index(drop=True)
    clf = LogisticRegression(
        penalty="l1", solver="liblinear", C=best_C,
        class_weight="balanced", max_iter=2000, random_state=b,
    )
    clf.fit(Xb, yb)
    bootstrap_picks.iloc[:, b] = (np.abs(clf.coef_[0]) > 1e-8).astype(int)

stability_score = bootstrap_picks.mean(axis=1).sort_values(ascending=False)
THRESHOLD_STABILITY = 0.60
selection_stability = stability_score[stability_score >= THRESHOLD_STABILITY].index.tolist()
print(f"stability LASSO @ {THRESHOLD_STABILITY}: {len(selection_stability)} features")
print((stability_score.head(15) * 100).round(1).to_string())


stability LASSO @ 0.6: 11 features
pos_A                      100.0
ratio_peak_speed           100.0
pos_D                      100.0
formation_offensiveness    100.0
is_home_int                 92.0
subbed_int                  90.0
ratio_shots_top_third       84.0
jersey_number               82.0
minute_in                   78.0
ratio_distance              76.0
cumul_mean_max_speed        62.0
cumul_shots_on_target       58.0
ratio_sprints               58.0
cumul_peak_speed            54.0
last15_peak_speed           50.0


In [11]:
"""BorutaPy with XGBoost — uses tree importance, no scoring metric."""
xgb_for_boruta = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    scale_pos_weight=spw, n_jobs=-1, random_state=RANDOM_SEED,
    tree_method="hist", verbosity=0,
)
boruta_selector = BorutaPy(
    xgb_for_boruta, n_estimators="auto", max_iter=50,
    random_state=RANDOM_SEED, verbose=0,
)
boruta_selector.fit(X_train_pre.values, y_train.values)
confirmed = [features_pre[i] for i in range(len(features_pre)) if boruta_selector.support_[i]]
tentative = [features_pre[i] for i in range(len(features_pre)) if boruta_selector.support_weak_[i]]
selection_boruta = confirmed + tentative
print(f"Boruta confirmed: {len(confirmed)} -- {confirmed}")
print(f"Boruta tentative: {len(tentative)} -- {tentative}")


Boruta confirmed: 6 -- ['pos_A', 'pos_D', 'cumul_sprints', 'jersey_number', 'cumul_peak_speed', 'subbed_int']
Boruta tentative: 5 -- ['ratio_peak_speed', 'formation_offensiveness', 'is_home_int', 'cumul_shots_on_target', 'cumul_shots_under_press']


In [12]:
"""RFE-CV with AP scoring (THE KEY CHANGE vs 02b)."""
xgb_for_rfe = xgb.XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1,
    scale_pos_weight=spw, n_jobs=-1, random_state=RANDOM_SEED,
    tree_method="hist", verbosity=0,
)
rfe = RFECV(
    estimator=xgb_for_rfe, step=2, min_features_to_select=5,
    cv=GroupKFold(5).split(X_train_pre, y_train, g_train),
    scoring="average_precision",     # ← KEY CHANGE
    n_jobs=1,
)
rfe.fit(X_train_pre, y_train)
selection_rfe_ap = [features_pre[i] for i in range(len(features_pre)) if rfe.support_[i]]
print(f"RFE-CV @ AP selected {rfe.n_features_} features:")
for f in selection_rfe_ap: print(f"  {f}")


RFE-CV @ AP selected 19 features:
  pos_A
  ratio_peak_speed
  pos_D
  cumul_hsr
  formation_offensiveness
  cumul_sprints
  jersey_number
  last15_shots_top_third
  is_home_int
  ratio_sprints
  last15_peak_speed
  cumul_shots_on_target
  cumul_distance
  cumul_peak_speed
  cumul_shots_under_press
  ratio_distance
  subbed_int
  cumul_mean_max_speed
  last15_distance


## Section D — Winner pick by OOF AP

For each candidate subset, train XGBoost with 5-fold GroupKFold and
report **OOF average precision** as the primary metric. We also report
OOF AUC for cross-reference but pick the winner by AP.


In [13]:
"""Voting + winner pick."""
votes = pd.DataFrame(0, index=features_pre,
                     columns=["lasso", "stability", "boruta", "rfe_ap"])
votes.loc[selection_lasso, "lasso"] = 1
votes.loc[selection_stability, "stability"] = 1
votes.loc[selection_boruta, "boruta"] = 1
votes.loc[selection_rfe_ap, "rfe_ap"] = 1
votes["total"] = votes.sum(axis=1)
selection_consensus_strong = votes[votes["total"] >= 3].index.tolist()
selection_consensus_soft = votes[votes["total"] >= 2].index.tolist()

SUBSETS = {
    "lasso": selection_lasso,
    "stability": selection_stability,
    "boruta": selection_boruta,
    "rfe_ap": selection_rfe_ap,
    "consensus_3+": selection_consensus_strong,
    "consensus_2+": selection_consensus_soft,
}


def cv_oof_xgb(X, y, groups, factory):
    oof = np.zeros(len(X))
    best_iters = []
    for tr, va in GroupKFold(5).split(X, y, groups):
        m = factory()
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va], y.iloc[va])], verbose=False)
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        best_iters.append(getattr(m, "best_iteration", m.n_estimators))
    return oof, best_iters


def factory_default(spw_):
    return lambda: xgb.XGBClassifier(
        n_estimators=600, learning_rate=0.05, max_depth=3,
        min_child_weight=10, subsample=0.9, colsample_bytree=0.85,
        scale_pos_weight=spw_,
        objective="binary:logistic", eval_metric="logloss",
        early_stopping_rounds=30, n_jobs=-1, random_state=RANDOM_SEED,
        tree_method="hist", verbosity=0,
    )


results = []
oof_per_subset = {}
for name, feats in SUBSETS.items():
    if len(feats) == 0:
        print(f"{name}: 0 features, skip"); continue
    oof, biters = cv_oof_xgb(X_train_pre[feats], y_train, g_train, factory_default(spw))
    auc = roc_auc_score(y_train, oof)
    ap = average_precision_score(y_train, oof)
    results.append({
        "subset": name, "n_features": len(feats),
        "oof_auc": auc, "oof_ap": ap,
        "median_n_estimators": int(np.median(biters)),
    })
    oof_per_subset[name] = oof
    print(f"  {name:14s}  k={len(feats):3d}  OOF AUC={auc:.4f}  AP={ap:.4f}")

results_df = pd.DataFrame(results).sort_values("oof_ap", ascending=False).reset_index(drop=True)
print()
print("ranked by OOF AP:")
print(results_df.to_string(index=False))


  lasso           k= 13  OOF AUC=0.6236  AP=0.1028
  stability       k= 11  OOF AUC=0.6182  AP=0.1095
  boruta          k= 11  OOF AUC=0.6321  AP=0.1059
  rfe_ap          k= 19  OOF AUC=0.6252  AP=0.1081
  consensus_3+    k=  8  OOF AUC=0.6242  AP=0.1048
  consensus_2+    k= 17  OOF AUC=0.6160  AP=0.1014

ranked by OOF AP:
      subset  n_features  oof_auc  oof_ap  median_n_estimators
   stability          11   0.6182  0.1095                  512
      rfe_ap          19   0.6252  0.1081                  592
      boruta          11   0.6321  0.1059                  373
consensus_3+           8   0.6242  0.1048                  421
       lasso          13   0.6236  0.1028                  582
consensus_2+          17   0.6160  0.1014                  477


In [14]:
"""Pick winner."""
winner = results_df.iloc[0]
WINNER_NAME = winner["subset"]
WINNER_FEATURES = SUBSETS[WINNER_NAME]
print(f"WINNER (by OOF AP): {WINNER_NAME}")
print(f"  features ({len(WINNER_FEATURES)}): {WINNER_FEATURES}")
print(f"  OOF AP: {winner['oof_ap']:.4f}")
print(f"  OOF AUC: {winner['oof_auc']:.4f}")


WINNER (by OOF AP): stability
  features (11): ['pos_A', 'ratio_peak_speed', 'pos_D', 'formation_offensiveness', 'is_home_int', 'subbed_int', 'ratio_shots_top_third', 'jersey_number', 'minute_in', 'ratio_distance', 'cumul_mean_max_speed']
  OOF AP: 0.1095
  OOF AUC: 0.6182


## Section E — SMOTE strategy comparison (judge by AP)

In [15]:
"""SMOTE comparison with AP."""
RESAMPLERS = {
    "vanilla": None,
    "ROS_0.2": RandomOverSampler(sampling_strategy=0.2, random_state=RANDOM_SEED),
    "ROS_0.3": RandomOverSampler(sampling_strategy=0.3, random_state=RANDOM_SEED),
    "SMOTE_0.2": SMOTE(sampling_strategy=0.2, k_neighbors=5, random_state=RANDOM_SEED),
    "SMOTE_0.3": SMOTE(sampling_strategy=0.3, k_neighbors=5, random_state=RANDOM_SEED),
    "SMOTETomek_0.3": SMOTETomek(sampling_strategy=0.3, random_state=RANDOM_SEED),
}


def oof_with_resampler(resampler, X, y, groups, params):
    oof = np.zeros(len(X))
    for tr, va in GroupKFold(5).split(X, y, groups):
        X_tr, y_tr = X.iloc[tr].copy(), y.iloc[tr].copy()
        X_va, y_va = X.iloc[va].copy(), y.iloc[va].copy()
        if resampler is not None:
            X_tr, y_tr = resampler.fit_resample(X_tr, y_tr)
        m = xgb.XGBClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        oof[va] = m.predict_proba(X_va)[:, 1]
    return oof


baseline_params = dict(
    n_estimators=600, learning_rate=0.05, max_depth=3,
    min_child_weight=10, subsample=0.9, colsample_bytree=0.85,
    scale_pos_weight=spw,
    objective="binary:logistic", eval_metric="aucpr",   # AP analogue
    early_stopping_rounds=30, n_jobs=-1, random_state=RANDOM_SEED,
    tree_method="hist", verbosity=0,
)

results_smote = []
for name, resampler in RESAMPLERS.items():
    params = dict(baseline_params)
    if resampler is not None:
        params["scale_pos_weight"] = 1.0
    oof_p = oof_with_resampler(resampler, X_train_pre[WINNER_FEATURES], y_train, g_train, params)
    auc = roc_auc_score(y_train, oof_p)
    ap = average_precision_score(y_train, oof_p)
    results_smote.append({"strategy": name, "oof_ap": ap, "oof_auc": auc})
    print(f"  {name:18s}  OOF AP={ap:.4f}  AUC={auc:.4f}")
results_smote_df = pd.DataFrame(results_smote).sort_values("oof_ap", ascending=False).reset_index(drop=True)
SMOTE_WINNER_NAME = results_smote_df.iloc[0]["strategy"]
SMOTE_WINNER = RESAMPLERS[SMOTE_WINNER_NAME]
print(f"\nSMOTE winner: {SMOTE_WINNER_NAME}")


  vanilla             OOF AP=0.1162  AUC=0.6093
  ROS_0.2             OOF AP=0.1227  AUC=0.6415
  ROS_0.3             OOF AP=0.1321  AUC=0.6132
  SMOTE_0.2           OOF AP=0.1245  AUC=0.5953
  SMOTE_0.3           OOF AP=0.1142  AUC=0.6089
  SMOTETomek_0.3      OOF AP=0.1151  AUC=0.6117

SMOTE winner: ROS_0.3


## Section F — Optuna with mean OOF AP as objective

In [16]:
"""Optuna with AP objective."""
def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 100, 800),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 6),
        min_child_weight=trial.suggest_float("min_child_weight", 1.0, 30.0),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        gamma=trial.suggest_float("gamma", 0.0, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 0.1, 5.0, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 0.0, 1.0),
        scale_pos_weight=spw if SMOTE_WINNER is None else 1.0,
        objective="binary:logistic", eval_metric="aucpr",
        early_stopping_rounds=30, n_jobs=-1, random_state=RANDOM_SEED,
        tree_method="hist", verbosity=0,
    )
    fold_aps = []
    for fold_idx, (tr, va) in enumerate(GroupKFold(5).split(X_train_pre[WINNER_FEATURES], y_train, g_train)):
        X_tr = X_train_pre[WINNER_FEATURES].iloc[tr].copy()
        y_tr = y_train.iloc[tr].copy()
        X_va = X_train_pre[WINNER_FEATURES].iloc[va].copy()
        y_va = y_train.iloc[va].copy()
        if SMOTE_WINNER is not None:
            X_tr, y_tr = SMOTE_WINNER.fit_resample(X_tr, y_tr)
        m = xgb.XGBClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        fold_aps.append(average_precision_score(y_va, m.predict_proba(X_va)[:, 1]))
        trial.report(np.mean(fold_aps), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(fold_aps))


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED, n_startup_trials=10),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2),
)
study.optimize(objective, n_trials=60, show_progress_bar=False, n_jobs=1)

print(f"finished {len(study.trials)} trials, "
      f"{sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)} pruned")
print(f"best OOF AP: {study.best_value:.4f}")
print(f"best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

OPTUNA_PARAMS = dict(study.best_params)
OPTUNA_PARAMS.update(dict(
    scale_pos_weight=spw if SMOTE_WINNER is None else 1.0,
    objective="binary:logistic", eval_metric="aucpr",
    early_stopping_rounds=30, n_jobs=-1, random_state=RANDOM_SEED,
    tree_method="hist", verbosity=0,
))


finished 60 trials, 0 pruned
best OOF AP: 0.1903
best params:
  n_estimators: 361
  learning_rate: 0.11048280926077925
  max_depth: 2
  min_child_weight: 28.750999485620444
  subsample: 0.9952454433996611
  colsample_bytree: 0.8194057192927462
  gamma: 0.04626821030076002
  reg_lambda: 0.16354185037103425
  reg_alpha: 0.7749603192567341


In [17]:
"""Build OOF with winning A+B config."""
oof_xgb_best = oof_with_resampler(
    SMOTE_WINNER, X_train_pre[WINNER_FEATURES], y_train, g_train, OPTUNA_PARAMS,
)
auc_AB = roc_auc_score(y_train, oof_xgb_best)
ap_AB = average_precision_score(y_train, oof_xgb_best)
print(f"After Section A+B+C+D+E+F: OOF AP = {ap_AB:.4f}  AUC = {auc_AB:.4f}")


After Section A+B+C+D+E+F: OOF AP = 0.1568  AUC = 0.6331


## Section G — F1-optimal thresholds (per position)

In [18]:
"""F1-optimal thresholds + isotonic calibration."""
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(oof_xgb_best, y_train)
oof_cal = iso.transform(oof_xgb_best)

THRESH_RANGE = np.linspace(0.005, 0.95, 200)


def best_threshold(probs, y, metric="f1"):
    if metric == "f1":
        scores = [f1_score(y, (probs >= t).astype(int), zero_division=0) for t in THRESH_RANGE]
    elif metric == "ba":
        scores = [balanced_accuracy_score(y, (probs >= t).astype(int)) for t in THRESH_RANGE]
    i = int(np.argmax(scores))
    return float(THRESH_RANGE[i]), float(scores[i])


# Global F1 threshold.
thr_f1_global, f1_oof = best_threshold(oof_cal, y_train, metric="f1")
thr_ba_global, ba_oof = best_threshold(oof_cal, y_train, metric="ba")
print(f"Global thresholds:")
print(f"  F1-optimal: {thr_f1_global:.3f} -> OOF F1 = {f1_oof:.4f}")
print(f"  BA-optimal: {thr_ba_global:.3f} -> OOF BA = {ba_oof:.4f}")
print()

# Per-position F1 thresholds.
percp_thresholds_f1 = {}
print("per-position F1 thresholds:")
for pos in ("A", "M", "D"):
    mask = train_pos == pos
    if mask.sum() < 50:
        percp_thresholds_f1[pos] = thr_f1_global; continue
    thr, f1v = best_threshold(oof_cal[mask], y_train.iloc[mask], metric="f1")
    percp_thresholds_f1[pos] = thr
    print(f"  pos={pos}  n={int(mask.sum()):4d}  thr={thr:.3f}  OOF F1={f1v:.4f}")


Global thresholds:
  F1-optimal: 0.100 -> OOF F1 = 0.2436
  BA-optimal: 0.071 -> OOF BA = 0.6238

per-position F1 thresholds:
  pos=A  n= 462  thr=0.100  OOF F1=0.3466
  pos=M  n= 995  thr=0.071  OOF F1=0.1553
  pos=D  n= 937  thr=0.048  OOF F1=0.0907


## Section H — Top-K + position filter (from 02f)

Apply the same precision-boosting post-processing as 02f to this
precision-first model.


In [19]:
"""Refit on full train + predict on test."""
final_params = dict(OPTUNA_PARAMS); final_params["early_stopping_rounds"] = None

X_train_resampled, y_train_resampled = X_train_pre[WINNER_FEATURES].copy(), y_train.copy()
if SMOTE_WINNER is not None:
    X_train_resampled, y_train_resampled = SMOTE_WINNER.fit_resample(
        X_train_pre[WINNER_FEATURES], y_train,
    )

model_xgb = xgb.XGBClassifier(**final_params)
model_xgb.fit(X_train_resampled, y_train_resampled, verbose=False)
test_proba_raw = model_xgb.predict_proba(X_test_pre[WINNER_FEATURES])[:, 1]

iso_final = IsotonicRegression(out_of_bounds="clip")
iso_final.fit(oof_xgb_best, y_train)
test_proba_cal = iso_final.transform(test_proba_raw)
oof_cal_final = iso_final.transform(oof_xgb_best)
print(f"raw test mean prob: {test_proba_raw.mean():.4f}")
print(f"cal test mean prob: {test_proba_cal.mean():.4f}")
print(f"actual test rate:    {y_test.mean():.4f}")


raw test mean prob: 0.1646
cal test mean prob: 0.0731
actual test rate:    0.0403


In [20]:
"""Multiple operating-point evaluations."""
def metrics_at(y_true, pred):
    return {
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred),
        "f1": f1_score(y_true, pred, zero_division=0),
        "BA": balanced_accuracy_score(y_true, pred),
        "n_pred_pos": int(pred.sum()),
    }


# Strategy 1: Per-position F1 threshold.
test_pred_perpos_f1 = np.zeros(len(test_proba_cal), dtype=int)
for pos, thr in percp_thresholds_f1.items():
    mask = test_pos == pos
    test_pred_perpos_f1[mask] = (test_proba_cal[mask] >= thr).astype(int)

# Strategy 2: Per-position BA threshold (for direct comparison with 02e).
percp_thresholds_ba = {}
for pos in ("A", "M", "D"):
    mask = train_pos == pos
    if mask.sum() < 50:
        percp_thresholds_ba[pos] = thr_ba_global; continue
    thr, _ = best_threshold(oof_cal[mask], y_train.iloc[mask], metric="ba")
    percp_thresholds_ba[pos] = thr
test_pred_perpos_ba = np.zeros(len(test_proba_cal), dtype=int)
for pos, thr in percp_thresholds_ba.items():
    mask = test_pos == pos
    test_pred_perpos_ba[mask] = (test_proba_cal[mask] >= thr).astype(int)

# Strategy 3: Top-K + A+M filter.
def combined_strategy(proba, fixture_ids, checkpoints, position_arr, k, threshold, allowed=("A", "M")):
    df = pd.DataFrame({
        "proba": proba, "fixture_id": fixture_ids, "checkpoint": checkpoints,
        "position": position_arr, "_idx": np.arange(len(proba)),
    })
    eligible = df["position"].isin(allowed) & (df["proba"] >= threshold)
    df["rank"] = df.loc[eligible].groupby(["fixture_id", "checkpoint"])["proba"].rank(method="first", ascending=False)
    df["pred"] = (eligible & (df["rank"] <= k)).fillna(False).astype(int)
    return df.sort_values("_idx").reset_index(drop=True)["pred"].values


test_fixtures = test["fixture_id"].values
test_pred_top5 = combined_strategy(test_proba_cal, test_fixtures, cp_test, test_pos, k=5, threshold=thr_f1_global)
test_pred_top3 = combined_strategy(test_proba_cal, test_fixtures, cp_test, test_pos, k=3, threshold=thr_f1_global)

print("=== TEST METRICS — 02g (precision-first pipeline) ===")
final_table = pd.DataFrame([
    {"strategy": "Per-pos F1 threshold", **metrics_at(y_test, test_pred_perpos_f1)},
    {"strategy": "Per-pos BA threshold", **metrics_at(y_test, test_pred_perpos_ba)},
    {"strategy": "Top-3 + A+M + F1 thr", **metrics_at(y_test, test_pred_top3)},
    {"strategy": "Top-5 + A+M + F1 thr", **metrics_at(y_test, test_pred_top5)},
])
print(final_table.round(4).to_string(index=False))

print()
print(f"  AUC (calibrated):  {roc_auc_score(y_test, test_proba_cal):.4f}")
print(f"  AP (calibrated):   {average_precision_score(y_test, test_proba_cal):.4f}")
print(f"  Brier (calibrated): {brier_score_loss(y_test, test_proba_cal):.4f}")


=== TEST METRICS — 02g (precision-first pipeline) ===
            strategy  precision  recall     f1     BA  n_pred_pos
Per-pos F1 threshold     0.0558  0.4483 0.0992 0.5649         233
Per-pos BA threshold     0.0553  0.4483 0.0985 0.5635         235
Top-3 + A+M + F1 thr     0.0750  0.2069 0.1101 0.5499          80
Top-5 + A+M + F1 thr     0.1224  0.4138 0.1890 0.6447          98

  AUC (calibrated):  0.7463
  AP (calibrated):   0.0822
  Brier (calibrated): 0.0489


## Section I — Side-by-side comparison: 02g vs 02e + 02f

Loads metrics from both pipelines and compares them on a common test
set (the same 6 held-out fixtures).


In [21]:
"""Load 02e + 02f metrics for comparison."""
ART_02e = PROJECT_ROOT / "models" / "advanced"
ART_02f = PROJECT_ROOT / "models" / "precision"
config_02e = json.loads((ART_02e / "config.json").read_text())
config_02f = json.loads((ART_02f / "config.json").read_text())

# 02e baseline (BA-optimised pipeline).
m_02e = config_02e["metrics"]
# 02f best (precision-optimised post-processing on 02e).
m_02f = config_02f["recommended_strategy"]["test_metrics"]

comparison = pd.DataFrame([
    {"pipeline": "02e (AUC-first + BA threshold)",
     "OOF_AUC": m_02e["oof_auc_section_AB"], "OOF_AP": float("nan"),
     "test_AUC": m_02e["test_auc"], "test_AP": float("nan"),
     "test_BA": m_02e["test_balanced_accuracy"],
     "test_precision": float("nan"), "test_recall": float("nan"), "test_F1": float("nan")},
    {"pipeline": "02e + 02f (top-5 + A+M + F1)",
     "OOF_AUC": m_02e["oof_auc_section_AB"], "OOF_AP": float("nan"),
     "test_AUC": m_02e["test_auc"], "test_AP": float("nan"),
     "test_BA": m_02f["balanced_accuracy"],
     "test_precision": m_02f["precision"], "test_recall": m_02f["recall"], "test_F1": m_02f["f1"]},
    {"pipeline": "02g (AP-first, per-pos F1 threshold)",
     "OOF_AUC": auc_AB, "OOF_AP": ap_AB,
     "test_AUC": roc_auc_score(y_test, test_proba_cal),
     "test_AP": average_precision_score(y_test, test_proba_cal),
     "test_BA": metrics_at(y_test, test_pred_perpos_f1)["BA"],
     "test_precision": metrics_at(y_test, test_pred_perpos_f1)["precision"],
     "test_recall": metrics_at(y_test, test_pred_perpos_f1)["recall"],
     "test_F1": metrics_at(y_test, test_pred_perpos_f1)["f1"]},
    {"pipeline": "02g + top-5 + A+M + F1",
     "OOF_AUC": auc_AB, "OOF_AP": ap_AB,
     "test_AUC": roc_auc_score(y_test, test_proba_cal),
     "test_AP": average_precision_score(y_test, test_proba_cal),
     "test_BA": metrics_at(y_test, test_pred_top5)["BA"],
     "test_precision": metrics_at(y_test, test_pred_top5)["precision"],
     "test_recall": metrics_at(y_test, test_pred_top5)["recall"],
     "test_F1": metrics_at(y_test, test_pred_top5)["f1"]},
])
print(comparison.round(4).to_string(index=False))


                            pipeline  OOF_AUC  OOF_AP  test_AUC  test_AP  test_BA  test_precision  test_recall  test_F1
      02e (AUC-first + BA threshold)   0.6933     NaN    0.7017      NaN   0.6824             NaN          NaN      NaN
        02e + 02f (top-5 + A+M + F1)   0.6933     NaN    0.7017      NaN   0.7113          0.1308       0.5862   0.2138
02g (AP-first, per-pos F1 threshold)   0.6331  0.1568    0.7463   0.0822   0.5649          0.0558       0.4483   0.0992
              02g + top-5 + A+M + F1   0.6331  0.1568    0.7463   0.0822   0.6447          0.1224       0.4138   0.1890


## Persist

In [22]:
"""Save."""
ART = PROJECT_ROOT / "models" / "precision_first"
ART.mkdir(exist_ok=True)

with open(ART / "model_xgb.pkl", "wb") as f:
    pickle.dump(model_xgb, f)
with open(ART / "isotonic_calibrator.pkl", "wb") as f:
    pickle.dump(iso_final, f)

X_train_pre[WINNER_FEATURES].to_csv(ART / "X_train_raw.csv", index=False)
X_test_pre[WINNER_FEATURES].to_csv(ART / "X_test_raw.csv", index=False)
y_train.to_csv(ART / "y_train_raw.csv", index=False, header=True)
y_test.to_csv(ART / "y_test_raw.csv", index=False, header=True)

oof_df = train[["player_appearance_id", "checkpoint", "fixture_id", "scored_after"]].copy()
oof_df["oof_proba"] = oof_xgb_best
oof_df["oof_proba_calibrated"] = oof_cal_final
oof_df.to_csv(ART / "oof_predictions.csv", index=False)

test_df_out = test[["player_appearance_id", "checkpoint", "fixture_id", "scored_after"]].copy()
test_df_out["test_proba"] = test_proba_raw
test_df_out["test_proba_calibrated"] = test_proba_cal
test_df_out["test_pred_perpos_f1"] = test_pred_perpos_f1
test_df_out["test_pred_top5_AM_f1"] = test_pred_top5
test_df_out.to_csv(ART / "test_predictions.csv", index=False)

results_df.to_csv(ART / "selection_comparison.csv", index=False)
results_smote_df.to_csv(ART / "smote_comparison.csv", index=False)
final_table.to_csv(ART / "final_metrics.csv", index=False)
comparison.to_csv(ART / "vs_02e_02f.csv", index=False)

config = {
    "winner_subset": WINNER_NAME,
    "winner_features": WINNER_FEATURES,
    "n_features": len(WINNER_FEATURES),
    "smote_winner": SMOTE_WINNER_NAME,
    "optuna_params": {k: float(v) if isinstance(v, (int, float, np.floating)) else v
                       for k, v in OPTUNA_PARAMS.items()},
    "f1_threshold_global": float(thr_f1_global),
    "ba_threshold_global": float(thr_ba_global),
    "percp_thresholds_f1": {k: float(v) for k, v in percp_thresholds_f1.items()},
    "percp_thresholds_ba": {k: float(v) for k, v in percp_thresholds_ba.items()},
    "metrics": {
        "oof_auc": float(auc_AB),
        "oof_ap": float(ap_AB),
        "test_auc": float(roc_auc_score(y_test, test_proba_cal)),
        "test_ap": float(average_precision_score(y_test, test_proba_cal)),
        "test_brier": float(brier_score_loss(y_test, test_proba_cal)),
    },
}
with open(ART / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"saved to {ART}/:")
for p in sorted(ART.iterdir()):
    print(f"  {p.name:40s} {p.stat().st_size / 1024:.1f} KB")


saved to c:\Users\tymot\projects\wec\models\precision_first/:
  config.json                              1.5 KB
  final_metrics.csv                        0.4 KB
  isotonic_calibrator.pkl                  0.8 KB
  model_xgb.pkl                            313.1 KB
  oof_predictions.csv                      138.2 KB
  selection_comparison.csv                 0.4 KB
  smote_comparison.csv                     0.3 KB
  test_predictions.csv                     35.3 KB
  vs_02e_02f.csv                           0.7 KB
  X_test_raw.csv                           37.6 KB
  X_train_raw.csv                          125.8 KB
  y_test_raw.csv                           2.1 KB
  y_train_raw.csv                          7.0 KB


### Summary

This pipeline differs from 02e in that **average precision** drives every
optimisation step (selection → SMOTE → hyperparams), instead of AUC.
The post-processing (top-K + A+M filter + F1 threshold) is the same as
02f. The notebook produces an **alternative deployment configuration**
that should be compared with 02e + 02f on a per-metric basis.

The honest expectation is a modest precision improvement (1-3
percentage points) — enough to validate the alternative pipeline
empirically, but not transformative. The fundamental ceiling of AUC ≈
0.76 on these features remains unchanged.
